# SPATIAL INTELLIGENCE - PART 1

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 5. Import the gallery floor plan

In [ ]:
gallery = Topology.ByBREPPath(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\gallery.brep")


## 6. Show the geometry

In [ ]:
Topology.Show(gallery,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Create a grid overlay

In [ ]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+3,3))
vRange = list(range(0,int(length)+3,3))

grid = Grid.EdgesByDistances(gallery, clip=True, uRange=uRange, vRange=vRange)

## 8. Show the geometry and the grid

In [ ]:
Topology.Show(gallery, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 9. Slice the floor plan with the grid to create a topologic shell

In [ ]:
shell = Topology.Slice(gallery, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## 10. Show the resulting shell

In [ ]:
Topology.Show(shell,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 11. Derive navigation and analysis graphs from the shell

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## 12. Derive and store the analysis graph vertices

In [ ]:
g_verts = Graph.Vertices(analysis_graph)

## 13. Show the analysis graph

In [ ]:
Topology.Show(analysis_graph,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

## 14. Spatial Intelligence through Graph Analysis

### a. Minimum Spanning Tree
MST is very time consuming to compute on a large grid graph. Here we will demonstrate it on a simple graph

In [ ]:
cc1 = CellComplex.Prism()
cc2 = Topology.Translate(cc1, 1.1, 0, 0)
cc3 = Topology.Translate(cc2, 1.1, 0, 0)
g1 = Graph.ByTopology(cc1)
g2 = Graph.ByTopology(cc2)
g3 = Graph.ByTopology(cc3)
g2 = Graph.MinimumSpanningTree(g2)
g3 = Graph.Complete(g3)
Topology.Show(g1, g2, g3,
              vertexSize=12,
              vertexColor="red",
              edgeColor="lightgrey",
              edgeWidth=4,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)


In [ ]:
dn1 = Graph.Density(g1)
dn2 = Graph.Density(g2)
dn3 = Graph.Density(g3)
print("Density 1:", dn1)
print("Density 2:", dn2)
print("Density 3:", dn3)

In [ ]:
dr1 = Graph.Diameter(g1)
dr2 = Graph.Diameter(g2)
dr3 = Graph.Diameter(g3)
print("Diameter 1:", dr1)
print("Diameter 2:", dr2)
print("Diameter 3:", dr3)

### b. Shortest Path (Use navigation graph)

In [ ]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=gallery)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

In [ ]:
Topology.Show(gallery, shortest_path, straight_path,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### c. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### d. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [ ]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

In [ ]:
status = Topology.ExportToBREP(gallery, path=r"C:\Users\sarwj\OneDrive - Cardiff University\Desktop\gallery.brep", overwrite=True)
print(status)